In [ ]:
#importing the libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import time
from datetime import datetime
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
import json
from torchsummary import summary

# Device will determine whether to run the valing on GPU or CPU.
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
display(device)

## Training Dataset Import

In [ ]:
#importing the dataset
data_s1 = pd.read_csv('../../../Data/Datasets/1. Sensor 1/Training Dataset/ml_dataset_01-30-2024_13-13.txt', header=0)
data_s2 = pd.read_csv('../../../Data/Datasets/2. Sensor 2/Training Dataset/combined_dataset_s2_08-23-2024_11-06.txt', header=0)
# data_s3 = pd.read_csv('../../../Data/Datasets/3. Sensor 3/Training Dataset/ml_dataset_v3_02-22-2024_16-10.txt', header=0)
data_s4 = pd.read_csv('../../../Data/Datasets/4. Sensor 4/Training Dataset/ml_dataset_s4_10-11-2024_11-14.txt', header=0)

data_s1 = data_s1.sample(frac=1)
data_s2 = data_s2.sample(frac=1)
# data_s3 = data_s3.sample(frac=1)
data_s4 = data_s4.sample(frac=1)

In [ ]:
data_s4.info

In [ ]:
# Ensure both datasets have the same number of columns
if data_s2.shape[1] == data_s4.shape[1]:
    # Rename columns to be the same across both datasets (ignoring original names)
    data_s4.columns = data_s2.columns

    # Combine datasets by position (row-wise concatenation)
    combined_data_s2_s4 = pd.concat([data_s2, data_s4], axis=0, ignore_index=True)

    # Now combine data_s1 with the combined_data_s2_s3 if they have the same column count
    if data_s1.shape[1] == combined_data_s2_s4.shape[1]:
        # Rename data_s1 columns to match the others
        data_s1.columns = combined_data_s2_s4.columns

        # Combine data_s1 with the combined data
        final_combined_data = pd.concat([data_s1, combined_data_s2_s4], axis=0, ignore_index=True)

        # Check the final combined data shape
        print(f"Final Combined Data Shape: {final_combined_data.shape}")  # Should be (total rows, 21)
    else:
        print(f"Error: Column count mismatch. Data_s1 has {data_s1.shape[1]} columns, Combined data has {combined_data_s2_s4.shape[1]} columns.")
else:
    print(f"Error: Column count mismatch. Data_s2 has {data_s2.shape[1]} columns, Data_s4 has {data_s4.shape[1]} columns.")

In [ ]:
final_combined_data.info

In [ ]:
input=final_combined_data.iloc[:,0:16].values
output=final_combined_data.iloc[:,16:20].values

In [ ]:
x_train = torch.Tensor(input)
y_train = torch.Tensor(output)

In [ ]:
# verify sizes
y_train.size()

In [ ]:
# Check for NaN in x_train
nan_indices_x = torch.isnan(x_train).any(dim=1)

# Remove rows with NaN from both x_train_normalized and y_train
x_train = x_train[~nan_indices_x]
y_train = y_train[~nan_indices_x]

# Check for NaN in y_train 
nan_indices_y = torch.isnan(y_train).any(dim=1)
x_train = x_train[~nan_indices_y]
y_train = y_train[~nan_indices_y]

x_train = x_train.view((x_train.shape[0], 4, 4, ))

In [ ]:
# Reshape to a 4x4 capacitive image
x_train = x_train.view((x_train.shape[0], 4, 4, ))

In [ ]:
# verify sizes
x_train.size()

## Open saved normalization statistics (fixed)

In [ ]:
# --- 1. Load Saved Normalization Statistics ---
with open("../../Data/Trained Models/LOGO-CV_Single_Contact_Estimation - Trained Models/normalization_statistics_fixed.json", "r") as f:
    stats = json.load(f)

# Convert to appropriate types for Torch
mean_input_train = stats["mean_input_train"]
std_input_train = stats["std_input_train"]
mean_y = torch.tensor(stats["mean_y"])
std_y = torch.tensor(stats["std_y"])

In [ ]:
input_transform = transforms.Normalize(mean=[mean_input_train], std=[std_input_train])
x_train_normalized = input_transform(x_train)

# --- Normalize Outputs using SAVED stats ---
# We only normalize the first 3 columns (X, Y, Force)
# Column 4 (Contact/No Contact) remains as 0s and 1s
y_coords_force = y_train[:, :3]
y_contact = y_train[:, 3].unsqueeze(1)

y_normalized_coords = (y_coords_force - mean_y) / std_y

# Recombine normalized coordinates with raw contact binary
y_train_normalized = torch.cat((y_normalized_coords, y_contact), dim=1)

# --- Verification ---
print(f"Final Input Shape: {x_train_normalized.shape}")
print(f"Final Output Shape: {y_train_normalized.shape}")

In [ ]:
y_train_normalized.size()

In [ ]:
x_train_normalized.size()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 8))
z = 10000
# Plot each output and colorbar
for i in range(4):
    row, col = divmod(i, 4)
    display(y_train[i+z].cpu())
    im = axes[i].imshow(x_train_normalized[i+z].cpu(), cmap='viridis')
    axes[i].set_title(f'Target Distribution {i+z}')
    axes[i].axis('off')

In [ ]:
x_train_normalized = x_train_normalized.to(device)
y_train_normalized = y_train_normalized.to(device)

In [ ]:
# Create PyTorch datasets and data loaders
train_dataset = TensorDataset(x_train_normalized, y_train_normalized)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=True)

## Load Model Architecture

In [ ]:
class CNNRegression(nn.Module):
    def __init__(self):
        super(CNNRegression, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(4)
        
        self.conv2 = nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(8)
        
        self.conv3 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(16)
        
        self.conv4 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.bn4 = nn.BatchNorm2d(32)

        self.conv5 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.bn5 = nn.BatchNorm2d(64)
        
        self.fc1 = nn.Linear(in_features=64*4*4, out_features=128) 
        self.fc2 = nn.Linear(in_features=128, out_features=4)
        
    def forward(self, x):
        x = x.unsqueeze(1)
        
        x = self.conv1(x)
        x = self.bn1(x)
        x = nn.functional.relu(x)
        
        x = self.conv2(x)
        x = self.bn2(x)
        x = nn.functional.relu(x)
        
        x = self.conv3(x)
        x = self.bn3(x)
        x = nn.functional.relu(x)
        
        x = self.conv4(x)
        x = self.bn4(x)
        x = nn.functional.relu(x)

        x = self.conv5(x)
        x = self.bn5(x)
        x = nn.functional.relu(x)
        
        x = x.view(x.size(0), -1)
        
        x = self.fc1(x)
        x = nn.functional.relu(x)
        
        x = self.fc2(x)
        
        return x

In [ ]:
# Instantiate the model
model = CNNRegression()

# Generate a random 4x4 input tensor
input_tensor = torch.rand(4, 4)

# Run one iteration
output_tensor = model(input_tensor.unsqueeze(0))  # Unsqueezing to add batch dimension

# Print the output tensor
print("Output tensor size:", output_tensor.size())
print("Output tensor:", output_tensor)

In [ ]:
model = CNNRegression()
model = model.to(device)

In [ ]:
# Print the summary of the model
summary(model, (4, 4))

In [ ]:
model.cuda

In [ ]:
data_parallel_model = torch.nn.DataParallel(model)

In [ ]:
#importing the validation dataset dataset

data_val = pd.read_csv('../../../Data/Datasets/3. Sensor 3/Validation Dataset/val_ml_dataset_s3_09-27-2024_16-08.txt', delim_whitespace=True)
# data_val = pd.read_csv('../../../Data/Datasets/4. Sensor 4/Validation Dataset/val_ml_dataset_sensorv4_02-27-2024_20-01.txt')
# data_val = pd.read_csv('../../../Data/Datasets/2. Sensor 2/Validation Dataset/validation_dataset_s2_08-12-2024_15-16_aligned.txt')
# data_val = pd.read_csv('../../../Data/Datasets/1. Sensor 1/Validation Dataset/val_ml_dataset_02-06-2024_15-17.txt')
data_val.info
data_val = data_val.sample(frac=1)
input_val =data_val.iloc[:,0:16].values
output_val=data_val.iloc[:,16:20].values
x_val = torch.Tensor(input_val)
y_val = torch.Tensor(output_val)

In [ ]:
data_val.info

In [ ]:
# Check for NaN in x_train
nan_indices_x = torch.isnan(x_val).any(dim=1)

# Remove rows with NaN from both x_train_normalized and y_val
x_val = x_val[~nan_indices_x]
y_val = y_val[~nan_indices_x]

# Check for NaN in y_train 
nan_indices_y = torch.isnan(y_val).any(dim=1)
x_val = x_val[~nan_indices_y]
y_val = y_val[~nan_indices_y]

In [ ]:
x_val = x_val.view((x_val.shape[0], 4, 4, ))
x_val = x_val.to(device)
y_val = y_val.to(device)
val_dataset = TensorDataset(x_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, drop_last=True)

## Run the training

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(data_parallel_model.parameters(), lr=0.001, weight_decay=1e-5)
reg_mse_loss = nn.MSELoss()
class_bce_loss = nn.BCEWithLogitsLoss()
reg_weight_location = 0.3
reg_weight_force = 0.5
reg_weight_classification = 1 - reg_weight_location - reg_weight_force
val_loss_x = []
val_loss_y = []
val_loss_force = []
# Train the model
for epoch in range(5):
    for i, (inputs, targets) in enumerate(train_loader):
        # Zero the gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs =  data_parallel_model(inputs)
        loss_regression_location = reg_mse_loss(outputs[:,0:2], targets[:,0:2])
        loss_regression_force = reg_mse_loss(outputs[:,2:3], targets[:,2:3])
        loss_classification = class_bce_loss(outputs[:,3:4],targets[:,3:4])

        # Backward pass and optimization
        combined_loss = reg_weight_location * loss_regression_location + reg_weight_force * loss_regression_force + reg_weight_classification * loss_classification
        combined_loss.backward()
        optimizer.step()

        # Print training statistics
        if (i + 1) % 10 == 0:
            print(f'Epoch [{epoch + 1}/{3}], Step [{i + 1}/{len(train_loader)}], Loss: {combined_loss.item():.4f}')

    # Validation loop
    with torch.no_grad():
        # Loop for calculating validation loss
        for name, idx, val_loss_list in [("X Location in mm", 0, val_loss_x), 
                                          ("Y Location in mm", 1, val_loss_y), 
                                          ("Force Magnitude in N", 2, val_loss_force)]:
            total_loss = 0
            total_instances = 0  # Total instances where the condition is met
            total_instances_with_condition = 0  # Total instances where the condition is met or not met
            
            # Loop through validation data
            for inputs, targets in val_loader:
                # Normalize inputs
                inputs_normalized = (inputs - mean_input_train) / std_input_train
                
                # Pass through model
                outputs = data_parallel_model(inputs_normalized)
                mean_y_gpu = mean_y.to(outputs.device)
                std_y_gpu = std_y.to(outputs.device)
                
                # Unnormalize outputs
                outputs_unnormalized = outputs[:, :3] * std_y_gpu + mean_y_gpu

                # Concatenate the contact/no-contact column of outputs with the unnormalized outputs
                outputs_unnormalized = torch.cat((outputs_unnormalized, outputs[:, 3].unsqueeze(1)), dim=1)
                
                # Calculate MAE for instances where the force condition is met
                condition_mask = targets[:, 2] < -0.02
                relevant_outputs = outputs_unnormalized[condition_mask]
                relevant_targets = targets[condition_mask]
                
                if relevant_outputs.size(0) > 0:
                    loss = torch.mean(torch.abs(relevant_outputs[:, idx] - relevant_targets[:, idx]))
                    total_loss += loss.item() * relevant_outputs.size(0)  # Accumulate the loss
                    total_instances_with_condition += relevant_outputs.size(0)
                    
                total_instances += targets.size(0)

            if total_instances_with_condition > 0:
                mean_loss = total_loss / total_instances_with_condition
                print(f'Val Loss for {name} (MAE): {mean_loss:.4f}')
                val_loss_list.append(mean_loss)  # Append the validation loss for the current epoch
            else:
                print(f'No instances found where the condition is met for {name}')
    
# Plotting the validation losses
epochs = range(1, 6) 
plt.plot(epochs, val_loss_x, label='X Location in mm')
plt.plot(epochs, val_loss_y, label='Y Location in mm')
plt.plot(epochs, val_loss_force, label='Force Magnitude in N')
plt.xlabel('Epochs')
plt.ylabel('Validation Loss (MAE)')
plt.title('Validation Loss for Different Outputs')
plt.legend()
plt.show()

In [ ]:
# Saving model 
torch.save(data_parallel_model.module.state_dict(),'../../../Data/LOGO-CV_Single_Point_Contact_Estimation_Results/Single_Point_Estimation_CNN_s1s2s4_'+ datetime.now().strftime("%Y%m%d-%H_%M") + '.pt' )
time.sleep(5)

### Evaluate on Test Dataset

In [ ]:
#importing the dataset
data_test = pd.read_csv('../../../Data/Datasets/3. Sensor 3/Validation Dataset/val_ml_dataset_s3_09-27-2024_16-08.txt', delim_whitespace=True)
# data_test = pd.read_csv('../../../Data/Datasets/4. Sensor 4/Validation Dataset/val_ml_dataset_sensorv4_02-27-2024_20-01.txt')
# data_test = pd.read_csv('../../../Data/Datasets/2. Sensor 2/Test Dataset/test_ml_dataset_sensorv2_02-09-2024_09-33')
# data_test = pd.read_csv('../../../Data/Datasets/1. Sensor 1/Test Dataset/test_ml_dataset_05-08-2024_17-24.txt')
data_test.info
data_test = data_test.sample(frac=1)
input_test =data_test.iloc[:,0:16].values
output_test = data_test.iloc[:,16:20].values
x_test = torch.Tensor(input_test)
y_test = torch.Tensor(output_test)
x_test.size()
# Check for NaN in x_train
nan_indices_x = torch.isnan(x_test).any(dim=1)

# Remove rows with NaN from both x_train_normalized and y_val
x_test = x_test[~nan_indices_x]
y_test = y_test[~nan_indices_x]

# Check for NaN in y_train 
nan_indices_y = torch.isnan(y_test).any(dim=1)
x_test = x_test[~nan_indices_y]
y_test = y_test[~nan_indices_y]
x_test = x_test.view((x_test.shape[0], 4, 4, ))
x_test = x_test.to(device)
y_test = y_test.to(device)
test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, drop_last=True)

dataframe = pd.DataFrame()
reg_mse_loss = nn.MSELoss(reduce=False)
reg_mae_loss = nn.L1Loss(reduce=False)
class_bce_loss = nn.BCEWithLogitsLoss(reduce=False)
m = nn.Sigmoid()

column_names = ["X_Target [mm]", "Y_Target [mm]", "Force_Target [N]", "Touch Classification Target", "X_Pred [mm]", "Y_Pred [mm]", "Force_Pred [N]", "Touch Classification Pred", "Error_X [mm]", "Error_Y [mm]", "Error_Force [N]", "Error_Classification"]

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs_normalized = (inputs - mean_input_train) / std_input_train
        outputs = data_parallel_model(inputs_normalized.float())
        mean_y_gpu = mean_y.to(outputs.device)
        std_y_gpu = std_y.to(outputs.device)    
        outputs_unnormalized = outputs.clone()
        outputs_unnormalized[:, :3] = outputs_unnormalized[:, :3] * std_y_gpu[:3] + mean_y_gpu[:3]
        x_loss = reg_mae_loss(outputs_unnormalized[:,0], targets[:,0])
        y_loss = reg_mae_loss(outputs_unnormalized[:,1], targets[:,1])
        force_loss = reg_mae_loss(outputs_unnormalized[:,2], targets[:,2])
        touch_loss = class_bce_loss(outputs[:,3], targets[:,3])
        outputs[:,3] = m(outputs[:,3])
        outputs_unnormalized[:,3] = outputs[:,3]
        data_row = torch.cat((targets, outputs_unnormalized, x_loss.unsqueeze(1), y_loss.unsqueeze(1), force_loss.unsqueeze(1), touch_loss.unsqueeze(1)), axis=1)
        dataframe_temp = pd.DataFrame(data_row.data.cpu())
        dataframe_temp.columns = column_names
        dataframe = pd.concat([dataframe, dataframe_temp])
dataframe.to_excel("../../../Data/Results/LOGO-CV_Single_Point_Contact_Estimation_Results/3. Train_s1s2s4_Test_s3/test_error_results_s3.xlsx", index=False)